# Thermal utilities: mass flow, temperature quality, exergy, and hydraulics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equinor/neqsim/blob/master/examples/notebooks/energy_networks/03_thermal_utilities_and_hydraulics.ipynb)

This notebook converts utility duty to real mass flow, checks temperature-grade feasibility, reports exergy, and screens a utility header for pressure drop and pump power.

**Implementation dependencies:** PRs #2610, #2612, and #2616.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = [Path(env_root).resolve()] if env_root else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_classes(neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True))
JClass = ns.JClass
print("NeqSim workspace classes loaded")

In [ ]:
UtilityEnergyBus = JClass("neqsim.process.equipment.energy.UtilityEnergyBus")
ThermalUtilityState = JClass("neqsim.process.equipment.energy.ThermalUtilityState")
ThermalUtilityQualityAnalysis = JClass("neqsim.process.equipment.energy.ThermalUtilityQualityAnalysis")
ThermalUtilityHeaderHydraulics = JClass("neqsim.process.equipment.energy.ThermalUtilityHeaderHydraulics")
UtilityLevel = JClass("neqsim.process.equipment.stream.UtilityLevel")

## 2. LP-steam duty and mass flow

In [ ]:
steam_supply = ThermalUtilityState(425.0, 4.0e5, 2.8e6)
condensate_return = ThermalUtilityState(383.0, 4.0e5, 0.6e6)
steam = UtilityEnergyBus("LP steam", UtilityLevel.LOW_PRESSURE_STEAM,
                         steam_supply, condensate_return)

duties_MW = [1, 2, 4, 6, 8, 10]
steam_rows = []
for duty in duties_MW:
    flow = steam.getMassFlowForDuty(duty, "MW", "kg/sec")
    exergy = ThermalUtilityQualityAnalysis.getExergyRateForDuty(steam, duty*1e6, 298.15)
    steam_rows.append({"duty_MW":duty, "steam_kg_s":flow, "exergy_MW":exergy/1e6})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
steam_df = pd.DataFrame(steam_rows)
steam_df

In [ ]:
ax = steam_df.plot(x="duty_MW", y="steam_kg_s", marker="o", figsize=(8,4))
ax.set_xlabel("Heating duty (MW)")
ax.set_ylabel("Steam flow (kg/s)")
ax.set_title("LP-steam consumption")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** The steam requirement is governed by the supply-to-return enthalpy drop, not merely by the utility label.

## 3. Temperature-grade feasibility

In [ ]:
process_temperatures = list(range(330, 431, 10))
feasibility = []
for process_T in process_temperatures:
    feasible = ThermalUtilityQualityAnalysis.canServeProcessTemperature(steam, process_T, 10.0)
    feasibility.append({"process_temperature_K":process_T, "feasible":int(feasible)})
feasibility_df = pd.DataFrame(feasibility)
feasibility_df

In [ ]:
ax = feasibility_df.plot(x="process_temperature_K", y="feasible", drawstyle="steps-post", figsize=(8,4))
ax.set_xlabel("Required process temperature (K)")
ax.set_ylabel("Feasible (1=yes)")
ax.set_title("LP-steam temperature-grade feasibility, ΔTmin = 10 K")
ax.set_ylim(-0.05, 1.05)
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Available MW cannot compensate for insufficient temperature grade. Infeasible heat allocations should be rejected before process convergence.

## 4. Cooling-water header hydraulics

In [ ]:
cooling_supply = ThermalUtilityState(293.15, 3.0e5, 84.0e3)
cooling_return = ThermalUtilityState(313.15, 2.5e5, 168.0e3)
cooling = UtilityEnergyBus("cooling water", UtilityLevel.COOLING_WATER,
                           cooling_supply, cooling_return)

header = ThermalUtilityHeaderHydraulics(
    500.0,   # length m
    0.35,    # inside diameter m
    4.5e-5,  # roughness m
    998.0,   # density kg/m3
    1.0e-3   # viscosity Pa s
)
header.setLocalLossCoefficient(12.0)
header.setPumpEfficiency(0.78)

hyd_rows = []
for duty in [2, 4, 6, 8, 10]:
    mass_flow = cooling.getMassFlowForDuty(duty, "MW", "kg/sec")
    result = header.calculate(mass_flow)
    hyd_rows.append({
        "duty_MW": duty,
        "mass_flow_kg_s": mass_flow,
        "velocity_m_s": result.getVelocity(),
        "pressure_drop_bar": result.getPressureDrop()/1e5,
        "pump_power_kW": result.getPumpPower()/1e3,
    })
hyd_df = pd.DataFrame(hyd_rows)
hyd_df

In [ ]:
ax = hyd_df.plot(x="mass_flow_kg_s", y=["pressure_drop_bar", "pump_power_kW"], marker="o", figsize=(8,4))
ax.set_xlabel("Cooling-water mass flow (kg/s)")
ax.set_ylabel("Pressure drop (bar) / pump power (kW)")
ax.set_title("Cooling-water header hydraulic response")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Pressure drop rises approximately with flow squared in turbulent service. Utility duty therefore creates a nonlinear pumping and capacity penalty.

## Summary

This workflow connects four layers:

1. thermal duty;
2. utility mass flow from enthalpy;
3. temperature-quality and exergy screening;
4. header pressure drop and pump power.

Detailed branched-network hydraulics and compressible steam flow remain specialist calculations.